# 03 - Terminal Cost and Terminal Set Geometry

Chapter 4.1 companion.

Goal: understand terminal cost, terminal controller, and terminal set geometry with visible computations.
        

In [ ]:
%matplotlib inline

import itertools
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_are

np.set_printoptions(precision=4, suppress=True)
        

## 1) Opening recap

- Terminal cost approximates the tail beyond the finite horizon.
- Terminal controller gives the local continuation law.
- Terminal set is the region where this continuation is admissible and trusted.
        

## 2) Stage 1: 1D microscope

Use x[k+1] = a x[k] + b u[k], u = -k x, and V_f(x) = p_inf x^2.
        

In [ ]:
a = 1.0
b = 1.0
q = 1.0
r = 1.0

p_inf = 0.5 * (q + np.sqrt(q ** 2 + 4.0 * q * r))
k_inf = (b * p_inf * a) / (r + b * b * p_inf)

print(f'p_inf = {p_inf:.10f}')
print(f'k_inf = {k_inf:.10f}')

levels = [0.5, 1.0, 2.0, 4.0]
interval_sizes = [np.sqrt(c / p_inf) for c in levels]

for c, rad in zip(levels, interval_sizes):
    print(f'V_f(x) <= {c:.1f}  ->  |x| <= {rad:.4f}')

plt.figure(figsize=(8, 3.5))
for idx, (c, rad) in enumerate(zip(levels, interval_sizes)):
    y = idx + 1
    plt.hlines(y, -rad, rad, linewidth=4, label=f'c = {c:g}')
    plt.plot([-rad, rad], [y, y], 'o')
plt.axvline(0.0, color='k', linewidth=0.8)
plt.yticks(range(1, len(levels) + 1), [f'level {i+1}' for i in range(len(levels))])
plt.xlabel('x')
plt.title('1D terminal-cost level intervals')
plt.grid(True, alpha=0.25)
plt.legend(loc='lower right')
plt.show()
        

## 3) 1D terminal set from input admissibility

Input constraint: |u| <= u_max and u = -k_inf x gives
|x| <= u_max / |k_inf|.

For X_f = {x : p_inf x^2 <= alpha_star}, we get
alpha_star = p_inf * (u_max / |k_inf|)^2.
        

In [ ]:
u_max_1d = 0.8
x_limit_1d = 5.0

x_adm = u_max_1d / abs(k_inf)
alpha_star_1d = p_inf * x_adm ** 2

print(f'Admissible interval from input bound: |x| <= {x_adm:.4f}')
print(f'alpha_star (1D) = {alpha_star_1d:.6f}')
assert alpha_star_1d > 0.0

x_axis = np.linspace(-x_limit_1d, x_limit_1d, 401)
xf_mask = p_inf * x_axis ** 2 <= alpha_star_1d

# Approximate N-step feasible intervals by brute force over a coarse input grid.
N_list = [1, 2, 4]
u_grid = np.linspace(-u_max_1d, u_max_1d, 51)
feasible_sets = {}

for N in N_list:
    feas = []
    for x0 in x_axis:
        found = False
        for U in itertools.product(u_grid, repeat=N):
            x = x0
            ok = True
            for uk in U:
                x = a * x + b * uk
                if abs(x) > x_limit_1d + 1e-9:
                    ok = False
                    break
            if ok and p_inf * x ** 2 <= alpha_star_1d + 1e-9:
                found = True
                break
        feas.append(found)
    feasible_sets[N] = np.array(feas)

plt.figure(figsize=(9, 4))
plt.fill_between(x_axis, -0.2, 0.2, where=np.ones_like(x_axis, dtype=bool), alpha=0.12, color='k', label='X')
for i, N in enumerate(N_list, start=1):
    y0 = i
    mask = feasible_sets[N]
    plt.scatter(x_axis[mask], y0 * np.ones(np.sum(mask)), s=8, alpha=0.45, label=f'X_N, N={N}')
plt.scatter(x_axis[xf_mask], (len(N_list) + 1) * np.ones(np.sum(xf_mask)), s=8, alpha=0.8, label='X_f')
plt.axvline(0.0, color='k', linewidth=0.8)
plt.ylim(0.5, len(N_list) + 1.5)
plt.yticks([1, 2, 3, 4], ['X_1', 'X_2', 'X_4', 'X_f'])
plt.xlabel('x')
plt.title('1D sets: X_f subset of larger feasible regions')
plt.grid(True, alpha=0.25)
plt.legend(loc='lower right')
plt.show()
        

### Observation

X_f is fixed by terminal ingredients.
As N increases, X_N expands.
The inclusion picture is X_f subset of X_N subset of X.
        

## 4) Stage 2: 2D double integrator

Now move the same ideas to
A = [[1,1],[0,1]], B = [[0],[1]].
        

In [ ]:
A = np.array([[1.0, 1.0],
              [0.0, 1.0]])
B = np.array([[0.0],
              [1.0]])
Q = np.diag([4.0, 1.0])
R = np.array([[0.2]])

P_inf_2d = solve_discrete_are(A, B, Q, R)
K_inf_2d = np.linalg.solve(R + B.T @ P_inf_2d @ B, B.T @ P_inf_2d @ A)

print('P_inf =')
print(P_inf_2d)
print('K_inf =')
print(K_inf_2d)

w, V = np.linalg.eigh(P_inf_2d)
print('Eigenvalues of P_inf:', w)
print('Columns of V are ellipse principal directions.')
        

In [ ]:
def ellipse_points_from_P(P, c, num=300):
    angles = np.linspace(0.0, 2.0 * np.pi, num)
    circle = np.vstack([np.cos(angles), np.sin(angles)])
    L = np.linalg.cholesky(P)
    pts = np.sqrt(c) * np.linalg.solve(L.T, circle)
    return pts

levels_2d = [1.0, 3.0, 6.0, 10.0]

plt.figure(figsize=(6, 6))
for c in levels_2d:
    pts = ellipse_points_from_P(P_inf_2d, c)
    plt.plot(pts[0, :], pts[1, :], label=f'x^T P x = {c:g}')

origin = np.zeros(2)
for i in range(2):
    direction = V[:, i] / np.sqrt(w[i])
    plt.arrow(origin[0], origin[1], direction[0], direction[1],
              width=0.01, head_width=0.08, length_includes_head=True)

plt.axhline(0.0, color='k', linewidth=0.8)
plt.axvline(0.0, color='k', linewidth=0.8)
plt.xlabel('position')
plt.ylabel('velocity')
plt.title('Terminal-cost level sets and eigen-directions')
plt.grid(True, alpha=0.25)
plt.axis('equal')
plt.legend()
plt.show()
        

### Observation

Larger eigenvalue means tighter direction of the ellipsoid.
Smaller eigenvalue means wider direction.
        

## 5) Descent funnel under terminal controller

Simulate u = -K_inf x and track V_f(x_k) = x_k^T P_inf x_k.
        

In [ ]:
steps = 25
x = np.zeros((steps + 1, 2))
u = np.zeros(steps)
V_values = np.zeros(steps + 1)

x[0] = np.array([3.0, -1.0])
V_values[0] = x[0] @ P_inf_2d @ x[0]

for k in range(steps):
    u[k] = float(-(K_inf_2d @ x[k]).item())
    x[k + 1] = A @ x[k] + B[:, 0] * u[k]
    V_values[k + 1] = x[k + 1] @ P_inf_2d @ x[k + 1]

print('V_f decreases in first steps:', np.all(np.diff(V_values[:12]) < 1e-9))

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
for c in [1.0, 3.0, 6.0, 10.0, 20.0]:
    pts = ellipse_points_from_P(P_inf_2d, c)
    plt.plot(pts[0, :], pts[1, :], color='0.75')
plt.plot(x[:, 0], x[:, 1], 'o-', label='LQR trajectory')
plt.xlabel('position')
plt.ylabel('velocity')
plt.title('Trajectory through terminal-cost level sets')
plt.grid(True, alpha=0.25)
plt.axis('equal')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(V_values, 'o-')
plt.xlabel('k')
plt.ylabel('V_f(x_k)')
plt.title('Value-function decrease')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()
        

## 6) Constraint-admissible terminal ellipsoid

Constraints:
- |position| <= x_max
- |velocity| <= v_max
- |u| <= u_max under u = -K_inf x

For faces F_i x <= b_i and ellipsoid x^T P x <= alpha:
max F_i x = sqrt(alpha * F_i P^{-1} F_i^T),
so alpha <= b_i^2 / (F_i P^{-1} F_i^T).
        

In [ ]:
x_max = 4.5
v_max = 2.5
u_max = 0.8

Krow = K_inf_2d.reshape(-1)

F_faces = np.array([
    [1.0, 0.0],
    [-1.0, 0.0],
    [0.0, 1.0],
    [0.0, -1.0],
    Krow,
    -Krow,
])

b_faces = np.array([x_max, x_max, v_max, v_max, u_max, u_max])

Pinv = np.linalg.inv(P_inf_2d)
alpha_candidates = []
for Fi, bi in zip(F_faces, b_faces):
    denom = Fi @ Pinv @ Fi.T
    alpha_i = (bi ** 2) / denom
    alpha_candidates.append(alpha_i)

alpha_star = float(0.995 * np.min(alpha_candidates))
print('alpha candidates:')
print(np.array(alpha_candidates))
print('alpha_star =', alpha_star)
assert alpha_star > 0.0

# Validate sampled points from X_f satisfy constraints.
angles = np.linspace(0.0, 2.0 * np.pi, 360)
boundary = ellipse_points_from_P(P_inf_2d, alpha_star, num=len(angles)).T
u_boundary = -(boundary @ Krow)

assert np.all(np.abs(boundary[:, 0]) <= x_max + 1e-9)
assert np.all(np.abs(boundary[:, 1]) <= v_max + 1e-9)
assert np.all(np.abs(u_boundary) <= u_max + 1e-6)
print('Check passed: sampled X_f boundary points satisfy state and input constraints.')
        

## 7) Plot terminal set X_f with box constraints
        

In [ ]:
pts_xf = ellipse_points_from_P(P_inf_2d, alpha_star)

plt.figure(figsize=(6, 6))
plt.axvline(x_max, color='k', linestyle=':')
plt.axvline(-x_max, color='k', linestyle=':')
plt.axhline(v_max, color='k', linestyle=':')
plt.axhline(-v_max, color='k', linestyle=':')
plt.plot(pts_xf[0, :], pts_xf[1, :], 'C1', linewidth=2.0, label='X_f: x^T P x <= alpha_star')
plt.plot(x[:, 0], x[:, 1], 'C0--', label='LQR trajectory')
plt.xlabel('position')
plt.ylabel('velocity')
plt.title('Terminal ellipsoid inside box and input-admissible region')
plt.grid(True, alpha=0.25)
plt.axis('equal')
plt.legend()
plt.show()
        

## 8) Approximate N-step feasible region X_N by coarse sampling

For each grid state x0, we test whether an input sequence of length N exists
such that:
- input and state bounds hold,
- terminal condition x_N in X_f holds.

This is a visualization by sampling, not a proof.
        

In [ ]:
N_test = 4
input_candidates = [-u_max, 0.0, u_max]

x1_grid = np.linspace(-x_max, x_max, 49)
x2_grid = np.linspace(-v_max, v_max, 41)

feasible_points = []
terminal_points = []

for x1 in x1_grid:
    for x2 in x2_grid:
        x0 = np.array([x1, x2])

        # Terminal set membership for direct comparison.
        in_xf = (x0 @ P_inf_2d @ x0) <= alpha_star + 1e-9
        if in_xf:
            terminal_points.append(x0)

        found = False
        for U_seq in itertools.product(input_candidates, repeat=N_test):
            xk = x0.copy()
            ok = True
            for uk in U_seq:
                if abs(uk) > u_max + 1e-9:
                    ok = False
                    break
                xk = A @ xk + B[:, 0] * uk
                if abs(xk[0]) > x_max + 1e-9 or abs(xk[1]) > v_max + 1e-9:
                    ok = False
                    break
            if ok and (xk @ P_inf_2d @ xk) <= alpha_star + 1e-9:
                found = True
                break

        if found:
            feasible_points.append(x0)

feasible_points = np.array(feasible_points)
terminal_points = np.array(terminal_points)

plt.figure(figsize=(7, 6))
plt.scatter(feasible_points[:, 0], feasible_points[:, 1], s=10, alpha=0.35, label=f'sampled X_N, N={N_test}')
plt.scatter(terminal_points[:, 0], terminal_points[:, 1], s=10, alpha=0.7, label='sampled X_f')
plt.axvline(x_max, color='k', linestyle=':')
plt.axvline(-x_max, color='k', linestyle=':')
plt.axhline(v_max, color='k', linestyle=':')
plt.axhline(-v_max, color='k', linestyle=':')
plt.xlabel('position')
plt.ylabel('velocity')
plt.title('Approximate feasible region X_N and terminal set X_f')
plt.grid(True, alpha=0.25)
plt.axis('equal')
plt.legend()
plt.show()
        

## 9) Honest general-case statement

In higher dimensions and with nonlinear constraints, exact terminal-set computation is nontrivial.

The core structure remains the same:
- terminal cost,
- terminal controller,
- terminal set.

Robust MPC adds disturbances and usually tightens sets further.
        